In [0]:
from pyspark.sql.functions import col, udf, pandas_udf
from pyspark.sql.types import StringType, IntegerType, DoubleType
import pandas as pd

data = [
    ("ravi kumar",    "Engineering", 55000, 28, "9876543210"),
    ("priya sharma",  "HR",          42000, 32, "8765432109"),
    ("arjun singh",   "Engineering", 72000, 26, "7654321098"),
    ("sneha gupta",   "Finance",     61000, 30, "invalid"),
    ("rohit verma",   "Engineering", 80000, 35, "9988776655"),
    ("meera nair",    "HR",          39000, 27, "ABC1234567"),
    ("karan mehta",   "Finance",     55000, 29, "8877665544"),
    ("divya pillai",  "Engineering", 91000, 33, "9123456789"),
]

cols = ["name", "dept", "salary", "age", "phone"]
df = spark.createDataFrame(data, cols)
display(df)

name,dept,salary,age,phone
ravi kumar,Engineering,55000,28,9876543210
priya sharma,HR,42000,32,8765432109
arjun singh,Engineering,72000,26,7654321098
sneha gupta,Finance,61000,30,invalid
rohit verma,Engineering,80000,35,9988776655
meera nair,HR,39000,27,ABC1234567
karan mehta,Finance,55000,29,8877665544
divya pillai,Engineering,91000,33,9123456789


In [0]:
# Problem: names are lowercase with full name
# Need: "ravi kumar" → "Ravi Kumar"
# Built-in initcap() exists — but let's learn UDF syntax first

# Step 1: Define regular Python function
def format_name(name):
    if name is None:
        return None
    return " ".join(word.capitalize() for word in name.split())

# Step 2: Register as UDF with return type
format_name_udf = udf(format_name, StringType())

# Step 3: Apply to column
df = df.withColumn("formatted_name",
    format_name_udf(col("name")))

display(df.select("name", "formatted_name"))

name,formatted_name
ravi kumar,Ravi Kumar
priya sharma,Priya Sharma
arjun singh,Arjun Singh
sneha gupta,Sneha Gupta
rohit verma,Rohit Verma
meera nair,Meera Nair
karan mehta,Karan Mehta
divya pillai,Divya Pillai


In [0]:
# Validate phone number — 10 digits only
# No built-in function for this → good UDF use case

def validate_phone(phone):
    if phone is None:
        return "Missing"
    if phone.isdigit() and len(phone) == 10:
        return "Valid"
    return "Invalid"

validate_phone_udf = udf(validate_phone, StringType())

df = df.withColumn("phone_status",
    validate_phone_udf(col("phone")))

display(df.select("name", "phone", "phone_status"))

name,phone,phone_status
ravi kumar,9876543210,Valid
priya sharma,8765432109,Valid
arjun singh,7654321098,Valid
sneha gupta,invalid,Invalid
rohit verma,9988776655,Valid
meera nair,ABC1234567,Invalid
karan mehta,8877665544,Valid
divya pillai,9123456789,Valid


In [0]:
# Calculate bonus based on dept and salary
# Complex logic — multiple conditions

def calculate_bonus(salary, dept):
    if salary is None or dept is None:
        return 0.0
    if dept == "Engineering":
        return round(salary * 0.20, 2)
    elif dept == "Finance":
        return round(salary * 0.15, 2)
    else:
        return round(salary * 0.10, 2)

# UDF with multiple input columns
from pyspark.sql.types import DoubleType
bonus_udf = udf(calculate_bonus, DoubleType())

df = df.withColumn("bonus",
    bonus_udf(col("salary"), col("dept")))

display(df.select("name", "dept", "salary", "bonus"))

name,dept,salary,bonus
ravi kumar,Engineering,55000,11000.0
priya sharma,HR,42000,4200.0
arjun singh,Engineering,72000,14400.0
sneha gupta,Finance,61000,9150.0
rohit verma,Engineering,80000,16000.0
meera nair,HR,39000,3900.0
karan mehta,Finance,55000,8250.0
divya pillai,Engineering,91000,18200.0


In [0]:
# Register UDF so you can use it in spark.sql() queries

spark.udf.register("validate_phone_sql", validate_phone, StringType())
spark.udf.register("calc_bonus_sql", calculate_bonus, DoubleType())

df.createOrReplaceTempView("employees")

# Now use in SQL
spark.sql("""
    SELECT
        name,
        dept,
        salary,
        phone,
        validate_phone_sql(phone) AS phone_status,
        calc_bonus_sql(salary, dept) AS bonus
    FROM employees
""").display()

name,dept,salary,phone,phone_status,bonus
ravi kumar,Engineering,55000,9876543210,Valid,11000.0
priya sharma,HR,42000,8765432109,Valid,4200.0
arjun singh,Engineering,72000,7654321098,Valid,14400.0
sneha gupta,Finance,61000,invalid,Invalid,9150.0
rohit verma,Engineering,80000,9988776655,Valid,16000.0
meera nair,HR,39000,ABC1234567,Invalid,3900.0
karan mehta,Finance,55000,8877665544,Valid,8250.0
divya pillai,Engineering,91000,9123456789,Valid,18200.0


In [0]:
# Pandas UDF processes entire column at once
# Much faster than row-by-row Python UDF
# Uses Apache Arrow for fast serialization

from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType, DoubleType
import pandas as pd

# Pandas UDF — takes pd.Series, returns pd.Series
@pandas_udf(StringType())
def format_name_pandas(names: pd.Series) -> pd.Series:
    return names.apply(
        lambda x: " ".join(w.capitalize()
        for w in x.split()) if x else None
    )

@pandas_udf(DoubleType())
def calc_bonus_pandas(
    salaries: pd.Series,
    depts: pd.Series
) -> pd.Series:
    result = []
    for salary, dept in zip(salaries, depts):
        if dept == "Engineering":
            result.append(round(salary * 0.20, 2))
        elif dept == "Finance":
            result.append(round(salary * 0.15, 2))
        else:
            result.append(round(salary * 0.10, 2))
    return pd.Series(result)

# Apply Pandas UDF
df = df.withColumn("name_clean",
    format_name_pandas(col("name")))

df = df.withColumn("bonus_pandas",
    calc_bonus_pandas(col("salary"), col("dept")))

display(df.select(
    "name", "name_clean", "dept", "salary", "bonus_pandas"
))

name,name_clean,dept,salary,bonus_pandas
ravi kumar,Ravi Kumar,Engineering,55000,11000.0
priya sharma,Priya Sharma,HR,42000,4200.0
arjun singh,Arjun Singh,Engineering,72000,14400.0
sneha gupta,Sneha Gupta,Finance,61000,9150.0
rohit verma,Rohit Verma,Engineering,80000,16000.0
meera nair,Meera Nair,HR,39000,3900.0
karan mehta,Karan Mehta,Finance,55000,8250.0
divya pillai,Divya Pillai,Engineering,91000,18200.0


In [0]:
# Show why built-in is always better when available

from pyspark.sql.functions import initcap, regexp_replace, length

# ❌ UDF way — slow
format_name_udf = udf(lambda x: x.title() if x else None, StringType())
df.withColumn("name_udf", format_name_udf(col("name"))).display()

# ✅ Built-in way — fast, Catalyst optimised
df.withColumn("name_builtin", initcap(col("name"))).display()

# ❌ UDF to check phone length
check_len_udf = udf(lambda x: len(x) if x else 0, IntegerType())
df.withColumn("ph_len", check_len_udf(col("phone"))).display()

# ✅ Built-in way
df.withColumn("ph_len", length(col("phone"))).display()

# ✅ Built-in regex for phone validation
from pyspark.sql.functions import when, regexp_extract
df.withColumn("ph_valid",
    when(col("phone").rlike("^[0-9]{10}$"), "Valid")
    .otherwise("Invalid")).display()

name,dept,salary,age,phone,formatted_name,phone_status,bonus,name_clean,bonus_pandas,name_udf
ravi kumar,Engineering,55000,28,9876543210,Ravi Kumar,Valid,11000.0,Ravi Kumar,11000.0,Ravi Kumar
priya sharma,HR,42000,32,8765432109,Priya Sharma,Valid,4200.0,Priya Sharma,4200.0,Priya Sharma
arjun singh,Engineering,72000,26,7654321098,Arjun Singh,Valid,14400.0,Arjun Singh,14400.0,Arjun Singh
sneha gupta,Finance,61000,30,invalid,Sneha Gupta,Invalid,9150.0,Sneha Gupta,9150.0,Sneha Gupta
rohit verma,Engineering,80000,35,9988776655,Rohit Verma,Valid,16000.0,Rohit Verma,16000.0,Rohit Verma
meera nair,HR,39000,27,ABC1234567,Meera Nair,Invalid,3900.0,Meera Nair,3900.0,Meera Nair
karan mehta,Finance,55000,29,8877665544,Karan Mehta,Valid,8250.0,Karan Mehta,8250.0,Karan Mehta
divya pillai,Engineering,91000,33,9123456789,Divya Pillai,Valid,18200.0,Divya Pillai,18200.0,Divya Pillai


name,dept,salary,age,phone,formatted_name,phone_status,bonus,name_clean,bonus_pandas,name_builtin
ravi kumar,Engineering,55000,28,9876543210,Ravi Kumar,Valid,11000.0,Ravi Kumar,11000.0,Ravi Kumar
priya sharma,HR,42000,32,8765432109,Priya Sharma,Valid,4200.0,Priya Sharma,4200.0,Priya Sharma
arjun singh,Engineering,72000,26,7654321098,Arjun Singh,Valid,14400.0,Arjun Singh,14400.0,Arjun Singh
sneha gupta,Finance,61000,30,invalid,Sneha Gupta,Invalid,9150.0,Sneha Gupta,9150.0,Sneha Gupta
rohit verma,Engineering,80000,35,9988776655,Rohit Verma,Valid,16000.0,Rohit Verma,16000.0,Rohit Verma
meera nair,HR,39000,27,ABC1234567,Meera Nair,Invalid,3900.0,Meera Nair,3900.0,Meera Nair
karan mehta,Finance,55000,29,8877665544,Karan Mehta,Valid,8250.0,Karan Mehta,8250.0,Karan Mehta
divya pillai,Engineering,91000,33,9123456789,Divya Pillai,Valid,18200.0,Divya Pillai,18200.0,Divya Pillai


name,dept,salary,age,phone,formatted_name,phone_status,bonus,name_clean,bonus_pandas,ph_len
ravi kumar,Engineering,55000,28,9876543210,Ravi Kumar,Valid,11000.0,Ravi Kumar,11000.0,10
priya sharma,HR,42000,32,8765432109,Priya Sharma,Valid,4200.0,Priya Sharma,4200.0,10
arjun singh,Engineering,72000,26,7654321098,Arjun Singh,Valid,14400.0,Arjun Singh,14400.0,10
sneha gupta,Finance,61000,30,invalid,Sneha Gupta,Invalid,9150.0,Sneha Gupta,9150.0,7
rohit verma,Engineering,80000,35,9988776655,Rohit Verma,Valid,16000.0,Rohit Verma,16000.0,10
meera nair,HR,39000,27,ABC1234567,Meera Nair,Invalid,3900.0,Meera Nair,3900.0,10
karan mehta,Finance,55000,29,8877665544,Karan Mehta,Valid,8250.0,Karan Mehta,8250.0,10
divya pillai,Engineering,91000,33,9123456789,Divya Pillai,Valid,18200.0,Divya Pillai,18200.0,10


name,dept,salary,age,phone,formatted_name,phone_status,bonus,name_clean,bonus_pandas,ph_len
ravi kumar,Engineering,55000,28,9876543210,Ravi Kumar,Valid,11000.0,Ravi Kumar,11000.0,10
priya sharma,HR,42000,32,8765432109,Priya Sharma,Valid,4200.0,Priya Sharma,4200.0,10
arjun singh,Engineering,72000,26,7654321098,Arjun Singh,Valid,14400.0,Arjun Singh,14400.0,10
sneha gupta,Finance,61000,30,invalid,Sneha Gupta,Invalid,9150.0,Sneha Gupta,9150.0,7
rohit verma,Engineering,80000,35,9988776655,Rohit Verma,Valid,16000.0,Rohit Verma,16000.0,10
meera nair,HR,39000,27,ABC1234567,Meera Nair,Invalid,3900.0,Meera Nair,3900.0,10
karan mehta,Finance,55000,29,8877665544,Karan Mehta,Valid,8250.0,Karan Mehta,8250.0,10
divya pillai,Engineering,91000,33,9123456789,Divya Pillai,Valid,18200.0,Divya Pillai,18200.0,10


name,dept,salary,age,phone,formatted_name,phone_status,bonus,name_clean,bonus_pandas,ph_valid
ravi kumar,Engineering,55000,28,9876543210,Ravi Kumar,Valid,11000.0,Ravi Kumar,11000.0,Valid
priya sharma,HR,42000,32,8765432109,Priya Sharma,Valid,4200.0,Priya Sharma,4200.0,Valid
arjun singh,Engineering,72000,26,7654321098,Arjun Singh,Valid,14400.0,Arjun Singh,14400.0,Valid
sneha gupta,Finance,61000,30,invalid,Sneha Gupta,Invalid,9150.0,Sneha Gupta,9150.0,Invalid
rohit verma,Engineering,80000,35,9988776655,Rohit Verma,Valid,16000.0,Rohit Verma,16000.0,Valid
meera nair,HR,39000,27,ABC1234567,Meera Nair,Invalid,3900.0,Meera Nair,3900.0,Invalid
karan mehta,Finance,55000,29,8877665544,Karan Mehta,Valid,8250.0,Karan Mehta,8250.0,Valid
divya pillai,Engineering,91000,33,9123456789,Divya Pillai,Valid,18200.0,Divya Pillai,18200.0,Valid


In [0]:
# Real pipeline: clean data using UDF where needed

# Custom logic that has NO built-in equivalent
def salary_band(salary, age):
    if salary is None or age is None:
        return "Unknown"
    if salary > 70000 and age < 30:
        return "High Potential"
    elif salary > 70000:
        return "Senior High"
    elif salary > 50000:
        return "Mid Level"
    else:
        return "Entry Level"

salary_band_udf = udf(salary_band, StringType())

df_final = df \
    .withColumn("name_clean", initcap(col("name"))) \
    .withColumn("phone_status",
        validate_phone_udf(col("phone"))) \
    .withColumn("salary_band",
        salary_band_udf(col("salary"), col("age"))) \
    .withColumn("bonus",
        bonus_udf(col("salary"), col("dept")))

display(df_final)

spark.sql("DROP TABLE IF EXISTS employees_enriched")
df_final.write.format("delta").mode("overwrite") \
    .saveAsTable("employees_enriched")
print("✅ Saved!")

name,dept,salary,age,phone,formatted_name,phone_status,bonus,name_clean,bonus_pandas,salary_band
ravi kumar,Engineering,55000,28,9876543210,Ravi Kumar,Valid,11000.0,Ravi Kumar,11000.0,Mid Level
priya sharma,HR,42000,32,8765432109,Priya Sharma,Valid,4200.0,Priya Sharma,4200.0,Entry Level
arjun singh,Engineering,72000,26,7654321098,Arjun Singh,Valid,14400.0,Arjun Singh,14400.0,High Potential
sneha gupta,Finance,61000,30,invalid,Sneha Gupta,Invalid,9150.0,Sneha Gupta,9150.0,Mid Level
rohit verma,Engineering,80000,35,9988776655,Rohit Verma,Valid,16000.0,Rohit Verma,16000.0,Senior High
meera nair,HR,39000,27,ABC1234567,Meera Nair,Invalid,3900.0,Meera Nair,3900.0,Entry Level
karan mehta,Finance,55000,29,8877665544,Karan Mehta,Valid,8250.0,Karan Mehta,8250.0,Mid Level
divya pillai,Engineering,91000,33,9123456789,Divya Pillai,Valid,18200.0,Divya Pillai,18200.0,Senior High


✅ Saved!
